# GPT-2 Medium: Accessibility Domain Knowledge

24 layers | 16 heads | d_model=1024 | d_mlp=4096 | sequential attn+MLP

In [1]:
import sys
sys.path.insert(0, '../../..')

import matplotlib.pyplot as plt
from src.models import load_model, unload
from src.logit_lens import logit_lens, print_logit_lens
from src.decompose import decompose_contributions, print_decomposition, summarize_contributions
from src.heads import per_head_contributions, top_heads, print_top_heads, print_layer_summary
from src.viz import plot_logit_lens, plot_decomposition, plot_head_heatmap, save_figures

In [3]:
model_name = "gpt2-medium"
model, info = load_model(model_name)

Loaded pretrained model gpt2-medium into HookedTransformer
Loaded gpt2-medium on mps
  24 layers | 16 heads | d_model=1024 | d_mlp=4096 | sequential attn+MLP


In [8]:
prompts = [
    ("The W in WCAG stands for", " Web", "wcag"),
    ("ARIA stands for Accessible Rich Internet", " Applications", "aria"),
    ("The purpose of alt text is to describe an", " image", "alt"),
    ("In HTML, the alt attribute provides a text description of an", " image", "HTML"),
    ("A blind person uses a screen", " reader", "screenReader"),
    ("A screen reader is", " software", "screenReaderv1"),
]

results = {}
for prompt, target, label in prompts:
    print(f"\n{'='*60}")
    print(f"  {label.upper()}: \"{prompt}\" → \"{target}\"")
    print(f"{'='*60}\n")

    df_lens, cache = logit_lens(model, prompt, target)
    print_logit_lens(df_lens)

    df_decomp = decompose_contributions(model, cache, target)
    print_decomposition(df_decomp)
    print(summarize_contributions(df_decomp))

    df_heads = per_head_contributions(model, cache, target)
    print_top_heads(df_heads)

    results[label] = {"lens": df_lens, "decomp": df_decomp, "heads": df_heads}


  WCAG: "The W in WCAG stands for" → " Web"

Model: gpt2-medium
Prompt: "The W in WCAG stands for"
Target: " Web" (token 5313)
Final prediction: " ""

Layer    Top Prediction       Target Rank    Target Prob 
------------------------------------------------------
0         the                 6712           0.000021    
1         the                 2927           0.000039    
2         example             5262           0.000022    
3         the                 10468          0.000007    
4         the                 14707          0.000003    
5         the                 7790           0.000009    
6         the                 2189           0.000050    
7         the                 809            0.000187    
8         '                   1147           0.000124    
9         the                 4352           0.000017    
10        the                 5490           0.000006    
11        the                 5160           0.000005    
12        acronym             2736     

In [9]:
for label, data in results.items():
    plt.close(plot_logit_lens(data["lens"]))
    plt.close(plot_decomposition(data["decomp"]))
    plt.close(plot_head_heatmap(data["heads"]))

In [10]:
for label, data in results.items():
    data["lens"].to_csv(f"../../results/gpt2/{model_name}/{label}-logit-lens.csv", index=False)
    data["decomp"].to_csv(f"../../results/gpt2/{model_name}/{label}-decomposition.csv", index=False)
    data["heads"].to_csv(f"../../results/gpt2/{model_name}/{label}-heads.csv", index=False)
    save_figures(model_name, label,
                 logit_lens=data["lens"], decomposition=data["decomp"], heads=data["heads"],
                 out_dir=f"../../results/gpt2/{model_name}/figures")

Saved: ../../results/gpt2/gpt2-medium/figures/wcag-logit-lens.png
Saved: ../../results/gpt2/gpt2-medium/figures/wcag-decomposition.png
Saved: ../../results/gpt2/gpt2-medium/figures/wcag-head-heatmap.png
Saved: ../../results/gpt2/gpt2-medium/figures/aria-logit-lens.png
Saved: ../../results/gpt2/gpt2-medium/figures/aria-decomposition.png
Saved: ../../results/gpt2/gpt2-medium/figures/aria-head-heatmap.png
Saved: ../../results/gpt2/gpt2-medium/figures/alt-logit-lens.png
Saved: ../../results/gpt2/gpt2-medium/figures/alt-decomposition.png
Saved: ../../results/gpt2/gpt2-medium/figures/alt-head-heatmap.png
Saved: ../../results/gpt2/gpt2-medium/figures/HTML-logit-lens.png
Saved: ../../results/gpt2/gpt2-medium/figures/HTML-decomposition.png
Saved: ../../results/gpt2/gpt2-medium/figures/HTML-head-heatmap.png
Saved: ../../results/gpt2/gpt2-medium/figures/screenReader-logit-lens.png
Saved: ../../results/gpt2/gpt2-medium/figures/screenReader-decomposition.png
Saved: ../../results/gpt2/gpt2-medium/fi

In [7]:
unload(model)

Model unloaded, memory cleared
